# Deep Network Development, 2026 Autumn - Week 8 Practice
## Transfer learning, regularization, and model evaluation

Dataset: M. Buscema, S. Terzi, and W. Tastle, [Steel Plates Faults](https://doi.org/10.24432/C5J88N), UCI Machine Learning Repository (2010), CC BY 4.0.

In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

def set_seed():
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

torch.set_num_threads(1)  # fast and CPU-friendly for this small dataset
set_seed()

## Provided setup: data and splits

### Your task:

1. Load the 'SteelPlatesFaults.csv'
2. Create two sets of samples, first contains only the first 3 classes the other uses the whole class list.
3. Use 'StandardScaler' to scale the dataset (be aware of information leakage, scale the two sets separately)

In [ ]:
df = pd.read_csv('SteelPlatesFaults.csv')
feature_names = df.columns[:27].tolist()
class_names = df.columns[27:].tolist()


## Provided setup: model and evaluation

The network uses `nn.Sequential` and returns raw scores (logits); `CrossEntropyLoss` combines the appropriate normalization and loss. In each training loop, early stopping monitors the **complete validation set**, saves the best weights, and restores them. The recorded epoch starts at 1.

In [ ]:
class FaultsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(27, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 3)
        )

    def forward(self, x):
        return self.model(x)

criterion = nn.CrossEntropyLoss()
BATCH_SIZE = 16
MAX_EPOCHS = 800
PATIENCE = 5
LEARNING_RATE = 0.1

def plot_losses(history, title):
    plt.figure(figsize=(6, 3))
    plt.plot(history['train'], label='Training classification loss')
    plt.plot(history['val'], label='Validation classification loss')
    plt.axvline(history['best_epoch'] - 1, color='grey', linestyle='--', label='Best epoch')
    plt.xlabel('Epoch (starting at 0)')
    plt.ylabel('Cross-entropy')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()
    print('Best validation epoch:', history['best_epoch'])

def evaluate(model, X, y, names, title):
    model.eval()
    with torch.no_grad():
        prediction = model(X).argmax(dim=1).numpy()
    target = y.numpy()
    scores = {'Accuracy': accuracy_score(target, prediction),
              'Balanced accuracy': balanced_accuracy_score(target, prediction),
              'Macro-F1': f1_score(target, prediction, labels=np.arange(len(names)), average='macro', zero_division=0)}
    print(title, {k: round(v, 3) for k, v in scores.items()})
    cm = confusion_matrix(target, prediction, labels=np.arange(len(names)))
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, xticks_rotation=90, colorbar=False, cmap='viridis')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return scores, cm

## Exercise 1: Baseline multiclass MLP

Train on Pastry, Z_Scratch, and K_Scatch with **full-batch gradient descent**: each update uses all baseline training examples. Run the provided code. Try a learning rate or patience value, using the baseline validation curve to guide your choice. Which classes does the confusion matrix mix up?

## Exercise 2: Mini-batch training

A mini-batch update uses a subset of training examples; one epoch visits all examples when sampling without replacement. This gives more frequent, noisier updates than full-batch training.

Your task is to:
1. Define the `Dataset` using the already split dataset. (For convenience use the `TensorDataset`, no need for custom dataset definition here)
2. Create a `DataLoader` using the dataset.
3. Run the training using the `DataLoader`, using Early Stopping (save the best model weights).
4. Plot the training and validation losses.
5. Load the best model weights, and run the evaluation on the test set. You can load the weights by `model.load_state_dict()` and get the copy of the current models state dictionary with `copy.deepcopy(mini_model.state_dict())`
6. Plot he confusion matrix, print the accuracy, balanced-accuracy and Macro F1 score.

## Exercise 3: Transfer learning and new classes

Your task is:

1. Create a **new model class** that contains a copy of the pretrained mini-batch model hidden layers.
2. Replace the last layer. The three-output baseline head is not used.
3. Freeze the first copied hidden layers and optimize only the newly added layers parameter.
4. Plot train and validation losses.
5. Evaluate the best model weights and calculate the metrics and plot the confusion matrix.

`NOTE: Reset the seed before dataset creation and training with the *set_seed()* function`

`NOTE: To freeze the model weights set the requires_grad to False. You can get the model parameters with the model.paramters() function.`

# Exercise 4

Your task is:

1. Create a **new model class** that can predict all the classes present in the dataset.
2. Train the model from scratch (no weight loading)
4. Plot train and validation losses.
5. Evaluate the best model weights and calculate the metrics and plot the confusion matrix.

`NOTE: Reset the seed before dataset creation and training with the *set_seed()* function`

## Exercise 5: Inspecting class imbalance

The cleaned file is sorted by class, but `train_test_split` shuffles and stratifies before separating the data. Look at the number of examples of each class in the training, validation, and test sets. 

*Which classes are least common?*

## Exercise 6: Balance the training dataset

Use `WeightedRandomSampler` to balace the dataset during training.

Sample **only from the training partition**. `X_train` is already transformed with the `StandardScaler` fitted on the training data. The balanced dataset reuses those scaled features. Validation and test stay fixed and retain their original class proportions. Train a fresh six-class model from scratch and compare its results with the unregularized reference.

`Note: Reset the seed before model inicialization and training`

## Exercise 7: Visualizing model weights

The first layer connects each input feature to hidden units. A larger absolute weight indicates a stronger individual connection, but does **not** by itself establish feature importance: scaling, other weights, and nonlinearities matter. Run the provided plots and change the selected model. Color limits and histogram bins are fixed across all weight plots. Feature labels F01–F27 correspond to the mapping in the data setup.

In [ ]:
WEIGHT_LIMIT = 1.5
WEIGHT_BINS = np.linspace(-WEIGHT_LIMIT, WEIGHT_LIMIT, 31)
NEAR_ZERO = 0.02

def first_weights(model):
    return model.model[0].weight.detach().numpy()

def plot_weights(model, title):
    w = first_weights(model)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    im = axes[0].imshow(w, aspect='auto', cmap='viridis', vmin=-WEIGHT_LIMIT, vmax=WEIGHT_LIMIT)
    axes[0].set_xticks(range(len(feature_labels)), feature_labels, rotation=90)
    axes[0].set_ylabel('Hidden unit')
    axes[0].set_title('First-layer weights')
    fig.colorbar(im, ax=axes[0])
    axes[1].hist(w.ravel(), bins=WEIGHT_BINS)
    axes[1].set_xlim(-WEIGHT_LIMIT, WEIGHT_LIMIT)
    axes[1].set_title('Weight distribution')
    axes[1].set_xlabel('Weight')
    axes[1].set_ylabel('Count')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    print('Mean absolute weight:', np.abs(w).mean(), 'Near zero:', (np.abs(w) < NEAR_ZERO).mean())

plot_weights(balanced_model, 'Balanced sampling')

## Exercise 8: Balanced sampling + L1 regularization

$$L_{\mathrm{total}} = L_{\mathrm{classification}} + \lambda_1 \sum_i |w_i|$$

**L1 — Specialists:** In a group project, a small number of people perform most of the work. Similarly, L1 encourages a model to rely on a smaller number of active connections. Here

- $L_{\mathrm{classification}}$ - the original loss that you use to train the model
- $\sum_i |w_i|$ - the sum of absolute values of the model weights (parameters)

### Your task:

- Use the scaled balanced training dataset from Exercise 5. In the mini-batch loop below, add the absolute-value penalty for **weights only**, not biases, to the classification loss before `backward()`. 
- Use $\lambda_1 = 0.005$ for the penalty. 
- Use the best model weights to calculate the metrics and create the confusion matrix

`Note: Reset the seed before model inicialization and training`

In [ ]:
L1_LAMBDA = 0.005



## Exercise 9: Balanced sampling + L2 regularization

$$L_{\mathrm{total}} = L_{\mathrm{classification}} + \lambda_2 \sum_i w_i^2$$

**L2 — Generalists:** In a group project, responsibility is distributed across more people, preventing any one person from dominating the workload. L2 discourages large weights; it does not require weights to be equal or nonzero. Here:

- $L_{\mathrm{classification}}$ - the original loss that you use to train the model
- $\sum_i w_i^2$ - the sum of square values of the model weights (parameters)


### Your task:

- Use the scaled balanced training dataset from Exercise 5. Add the sum of squared **weights**, excluding biases, to the classification loss in the mini-batch loop before calling `backward()`. 
- Use $\lambda_2 = 0.005$ for the penalty.

In [ ]:
L2_LAMBDA = 0.005



## Exercise 10: Comparing model weights

Compare the first layer of the balanced model with the balanced + L1 and balanced + L2 models. These models start from the same seeded weights and use the same scaled balanced training dataset. L1 encourages sparsity and L2 generally shrinks weights; these are tendencies, not guarantees for this experiment.

**Compared with the balanced model, how did L1 and L2 affect the first-layer weights?**

## Exercise 11: Final comparison

Plot the metrics of each model starting from the transfer learning model.

1. **Which models perform best? Inspect the confusion matrices and the metrics.**
2. **Does the highest accuracy also have the highest macro-F1? How does balanced training affect the confusion matrix?**

Some test classes have relatively few examples, so avoid broad claims from one split. Choose hyperparameters using validation data and evaluate the test set after choices are fixed.